### 1. Data Preparation

In [1]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.ticker as ticker
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import sqlite3 as sql

# Mengambil lokasi folder tempat file python/notebook ini berada
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

path = os.path.join(BASE_DIR, "retail.db")

# Menghubungkan ke database
conn = sql.connect(path)
conn = sql.connect(path)
cur= conn.cursor()
cur.execute("""
            SELECT
              name
            FROM sqlite_master
            WHERE type= 'table'
            """)
print(cur.fetchall())

[('Customers',), ('ProductCategories',), ('Products',), ('Stores',), ('Time',), ('Transactions',)]


### Data Extraction

In [12]:
query = """
SELECT
    -- Transaction
    t.TransactionID,

    -- Time
    tm.Date,
    tm.Month,
    tm.Quarter,
    tm.Year,

    -- Product
    p.ProductID,
    p.ProductName,

    -- Category
    pc.CategoryID,
    pc.CategoryName,

    -- Customer
    c.CustomerID,
    c.Gender,
    c.Age,

    -- Store
    s.StoreID,
    s.StoreName,
    s.City,
    s.Country,

    -- Sales
    t.Quantity,
    p.PricePerUnit,
    t.TotalAmount,
    t.DiscountApplied,

    -- Derived Metric
    (t.Quantity * p.PricePerUnit) AS GrossAmount

FROM Transactions AS t

LEFT JOIN Products AS p
    ON t.ProductID = p.ProductID

LEFT JOIN ProductCategories AS pc
    ON p.CategoryID = pc.CategoryID

LEFT JOIN Customers AS c
    ON t.CustomerID = c.CustomerID

LEFT JOIN Time AS tm
    ON t.TimeID = tm.TimeID

LEFT JOIN Stores AS s
    ON t.StoreID = s.StoreID;

"""
df = pd.read_sql_query(query, conn)
df.head()

,TransactionID,Date,Month,Quarter,Year,ProductID,ProductName,CategoryID,CategoryName,CustomerID,...,Age,StoreID,StoreName,City,Country,Quantity,PricePerUnit,TotalAmount,DiscountApplied,GrossAmount
0,1,2023-11-24,11,4,2023,P003,Perfume,1,Beauty,CUST001,...,34.0,S004,Medan Branch,Medan,Indonesia,3.0,50.0,150.0,12.87,150.0
1,2,2023-02-27,2,1,2023,P010,Suit,2,Clothing,CUST002,...,26.0,S004,Medan Branch,Medan,Indonesia,NaN,500.0,1000.0,13.95,NaN
2,3,2023-01-13,1,1,2023,P012,Charger,3,Electronics,CUST003,...,50.0,S004,Medan Branch,Medan,Indonesia,1.0,30.0,30.0,15.86,30.0
3,4,2023-05-21,5,2,2023,P010,Suit,2,Clothing,CUST004,...,37.0,S002,Surabaya Outlet,Surabaya,Indonesia,1.0,500.0,500.0,12.30,500.0
4,5,2023-05-06,5,2,2023,P003,Perfume,1,Beauty,CUST005,...,30.0,S005,Bali Shop,Bali,Indonesia,2.0,50.0,100.0,8.78,100.0


In [13]:
# ==========================================
# 1. STATISTIK DATA SEBELUM CLEANING
# ==========================================

print("=== DATASET OVERVIEW ===")
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom: {df.shape[1]:,}")

print("\n=== COLUMN INFORMATION ===")
print(df.info())

print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing": missing,
    "Missing (%)": missing_pct
}).sort_values("Missing", ascending=False)

print(missing_summary)

print("\n=== DUPLICATE ROWS ===")
print(f"Jumlah duplicate: {df.duplicated().sum():,}")

print("\n=== NUMERICAL STATISTICS ===")
display(df.describe().T)

=== DATASET OVERVIEW ===
Jumlah baris : 1,000
Jumlah kolom: 21

=== COLUMN INFORMATION ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    1000 non-null   object 
 1   Date             1000 non-null   object 
 2   Month            1000 non-null   int64  
 3   Quarter          1000 non-null   int64  
 4   Year             1000 non-null   int64  
 5   ProductID        1000 non-null   object 
 6   ProductName      1000 non-null   object 
 7   CategoryID       1000 non-null   int64  
 8   CategoryName     1000 non-null   object 
 9   CustomerID       1000 non-null   object 
 10  Gender           947 non-null    object 
 11  Age              954 non-null    float64
 12  StoreID          1000 non-null   object 
 13  StoreName        1000 non-null   object 
 14  City             1000 non-null   object 
 15  Country         

,count,mean,std,min,25%,50%,75%,max
Month,1000.0,6.549000,3.452755,1.0,4.00,6.00,10.000,12.0
Quarter,1000.0,2.525000,1.119208,1.0,2.00,2.00,4.000,4.0
Year,1000.0,2023.002000,0.044699,2023.0,2023.00,2023.00,2023.000,2024.0
CategoryID,1000.0,2.035000,0.805247,1.0,1.00,2.00,3.000,3.0
Age,954.0,41.382600,13.666595,18.0,29.00,42.00,53.000,64.0
Quantity,700.0,2.518571,1.131395,1.0,1.00,3.00,4.000,4.0
PricePerUnit,1000.0,179.890000,189.681356,25.0,30.00,50.00,300.000,500.0
TotalAmount,1000.0,456.000000,559.997632,25.0,60.00,135.00,900.000,2000.0
DiscountApplied,947.0,10.167814,5.832970,0.0,5.14,10.26,15.185,20.0
GrossAmount,700.0,447.450000,552.293790,25.0,60.00,120.00,900.000,2000.0


In [21]:
# 2. CLEANING

median_quantity = df['Quantity'].median()
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
df['GrossAmount'] = df['GrossAmount'].fillna(df['TotalAmount'])
df['Quantity'] = df['Quantity'].fillna(median_quantity)
df['DiscountApplied'] = df['DiscountApplied'].fillna(df.groupby('ProductID')['DiscountApplied'].transform('median'))
df = df.dropna(subset=['Gender']).reset_index(drop=True)


print("=== HASIL CLEANING ===")
print(f"Jumlah baris setelah cleaning : {df.shape[0]:,}")
print(f"Jumlah kolom                  : {df.shape[1]:,}")

=== HASIL CLEANING ===
Jumlah baris setelah cleaning : 947
Jumlah kolom                  : 21


In [22]:
# ==========================================
# 3. VALIDASI SETELAH CLEANING
# ==========================================

print("=== MISSING VALUES SETELAH CLEANING ===")
print(df.isnull().sum())

print("\n=== DUPLICATE SETELAH CLEANING ===")
print(f"Jumlah duplicate: {df.duplicated().sum():,}")

print("\n=== DATASET FINAL ===")
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom: {df.shape[1]:,}")

=== MISSING VALUES SETELAH CLEANING ===
TransactionID      0
Date               0
Month              0
Quarter            0
Year               0
ProductID          0
ProductName        0
CategoryID         0
CategoryName       0
CustomerID         0
Gender             0
Age                0
StoreID            0
StoreName          0
City               0
Country            0
Quantity           0
PricePerUnit       0
TotalAmount        0
DiscountApplied    0
GrossAmount        0
dtype: int64

=== DUPLICATE SETELAH CLEANING ===
Jumlah duplicate: 0

=== DATASET FINAL ===
Jumlah baris : 947
Jumlah kolom: 21


In [25]:
# Export CSV
df.to_csv("retail_dashboard_data.csv", index=False)

# Verifikasi
print("CSV berhasil diexport.")
print(f"Jumlah baris  : {df.shape[0]:,}")
print(f"Jumlah kolom  : {df.shape[1]:,}")
print(f"Nama file     : retail_dashboard_data.csv")

CSV berhasil diexport.
Jumlah baris  : 947
Jumlah kolom  : 21
Nama file     : retail_dashboard_data.csv


In [24]:
df.head(25)

,TransactionID,Date,Month,Quarter,Year,ProductID,ProductName,CategoryID,CategoryName,CustomerID,...,Age,StoreID,StoreName,City,Country,Quantity,PricePerUnit,TotalAmount,DiscountApplied,GrossAmount
0,1,2023-11-24,11,4,2023,P003,Perfume,1,Beauty,CUST001,...,34.0,S004,Medan Branch,Medan,Indonesia,3.0,50.0,150.0,12.87,150.0
1,2,2023-02-27,2,1,2023,P010,Suit,2,Clothing,CUST002,...,26.0,S004,Medan Branch,Medan,Indonesia,3.0,500.0,1000.0,13.95,1000.0
2,4,2023-05-21,5,2,2023,P010,Suit,2,Clothing,CUST004,...,37.0,S002,Surabaya Outlet,Surabaya,Indonesia,1.0,500.0,500.0,12.30,500.0
3,5,2023-05-06,5,2,2023,P003,Perfume,1,Beauty,CUST005,...,30.0,S005,Bali Shop,Bali,Indonesia,2.0,50.0,100.0,8.78,100.0
4,6,2023-04-25,4,2,2023,P002,Face Cream,1,Beauty,CUST006,...,45.0,S001,Main Store Jakarta,Jakarta,Indonesia,1.0,30.0,30.0,0.47,30.0
5,8,2023-02-22,2,1,2023,P011,Headphones,3,Electronics,CUST008,...,30.0,S003,Bandung Store,Bandung,Indonesia,4.0,25.0,100.0,9.68,100.0
6,9,2023-12-13,12,4,2023,P014,Laptop,3,Electronics,CUST009,...,63.0,S004,Medan Branch,Medan,Indonesia,3.0,300.0,600.0,19.69,600.0
7,10,2023-10-07,10,4,2023,P008,Jacket,2,Clothing,CUST010,...,52.0,S005,Bali Shop,Bali,Indonesia,3.0,50.0,200.0,1.45,200.0
8,11,2023-02-14,2,1,2023,P008,Jacket,2,Clothing,CUST011,...,23.0,S003,Bandung Store,Bandung,Indonesia,2.0,50.0,100.0,9.73,100.0
9,12,2023-10-30,10,4,2023,P001,Lipstick,1,Beauty,CUST012,...,35.0,S005,Bali Shop,Bali,Indonesia,3.0,25.0,75.0,12.58,75.0
